# Bot detection challenge

Предсказываем вероятность того, что кука — бот, по её событиям за суточное окно.
Метрика — Precision при Recall ≥ 0.70 (`metric.py`).

1. Загрузка, EDA, дубликаты
2. Окно наблюдения и `captcha_shown`
3. Признаки: поведение куки и общие объявления
4. Валидация и выбор модели
5. Важность признаков
6. Сабмит
7. Описание подхода

In [1]:
import sys
import numpy as np
import pandas as pd
import sklearn

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

print("python  :", sys.version.split()[0])
print("numpy   :", np.__version__)
print("pandas  :", pd.__version__)
print("sklearn :", sklearn.__version__)

python  : 3.14.4
numpy   : 2.5.1
pandas  : 3.0.3
sklearn : 1.9.0


In [2]:
train = pd.read_csv(
    "data/train.csv",
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
)
test = pd.read_csv(
    "data/test.csv",
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
)
events = pd.read_csv("data/events.csv.gz", parse_dates=["event_ts"])

print("train:", train.shape, " test:", test.shape, " events:", events.shape)
print("доля ботов в train:", round(train.target.mean(), 4))
print(train.head(3).to_string())

train: (11091, 5)  test: (4909, 4)  events: (328905, 14)
доля ботов в train: 0.0811
             cookie_id   cookie_created_at window_start_ts window_end_ts  target
0  ck_54a059eb7d3ea68b 2025-11-21 09:30:41      2026-04-06    2026-04-07       0
1  ck_7e4de46eeab82974 2025-09-23 10:10:24      2026-04-06    2026-04-07       0
2  ck_9320229ef6304522 2026-03-04 00:08:02      2026-04-06    2026-04-07       0


## 1. EDA

Типы событий, платформы (регистр разный — нужна нормализация), пропуски, окна train/test.

In [3]:
print(events.event_name.value_counts(), "\n")
print(events.platform.value_counts(), "\n")
print("пропуски по колонкам:")
print(events.isna().mean().round(3))

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64 

пропуски по колонкам:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
dtype: float64


In [4]:
print("окна train:", train.window_start_ts.min(), "..", train.window_start_ts.max())
print("окна test :", test.window_start_ts.min(), "..", test.window_start_ts.max())

окна train: 2026-04-06 00:00:00 .. 2026-04-19 00:00:00
окна test : 2026-04-20 00:00:00 .. 2026-04-26 00:00:00


Тест идёт строго после трейна по времени, поэтому валидироваться будем тоже по
времени, а не случайным сплитом.

### Дубликаты

В событиях есть полностью одинаковые строки — похоже на дубли записи в лог. Они
завышают счётчики и дают нулевые интервалы между событиями, поэтому удаляем их.

In [5]:
dup_mask = events.duplicated()
print("полных дублей строк во всех events:", dup_mask.sum(), "из", len(events))
print("затронутых cookie_id:", events.loc[dup_mask, "cookie_id"].nunique())

events = events.drop_duplicates().reset_index(drop=True)
print("events после dedup:", events.shape)

полных дублей строк во всех events: 4863 из 328905
затронутых cookie_id: 3775
events после dedup: (324042, 14)


## 2. Окно наблюдения

Признаки должны быть доступны на конец окна, поэтому берём только события с
`window_start_ts <= event_ts < window_end_ts`. Отдельно проверим капчу — кажется,
что это хороший признак бота.

In [6]:
m = events.merge(
    train[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id"
)
captcha = m[m.eid == 900]

print("captcha_shown у train-куки всего      :", len(captcha))
print(
    "  до начала окна                       :",
    (captcha.event_ts < captcha.window_start_ts).sum(),
)
print(
    "  внутри окна                          :",
    (
        (captcha.event_ts >= captcha.window_start_ts)
        & (captcha.event_ts < captcha.window_end_ts)
    ).sum(),
)
print(
    "  после конца окна                     :",
    (captcha.event_ts >= captcha.window_end_ts).sum(),
)
print(
    "доля ботов среди куки с captcha_shown  :",
    round(
        train.set_index("cookie_id").loc[captcha.cookie_id.unique(), "target"].mean(), 3
    ),
    " (базовая доля:",
    round(train.target.mean(), 3),
    ")",
)

captcha_shown у train-куки всего      : 7831
  до начала окна                       : 0
  внутри окна                          : 0
  после конца окна                     : 7831
доля ботов среди куки с captcha_shown  : 0.771  (базовая доля: 0.081 )


Капча ни разу не попадает в окно — она всегда после его конца, и у 77% таких
кук target = 1 (при 8% в среднем). То есть это реакция антибота уже после
окна — информация из будущего. Не используем.

In [7]:
def filter_window(events, meta):
    # события строго внутри [window_start_ts, window_end_ts)
    ev = events.merge(
        meta[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id"
    )
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]


ev_tr = filter_window(events, train)
ev_te = filter_window(events, test)
print("событий в окне: train =", len(ev_tr), " test =", len(ev_te))
print(
    "доля событий train-куки вне окна:",
    round(1 - len(ev_tr) / events.cookie_id.isin(train.cookie_id).sum(), 3),
)

событий в окне: train = 195484  test = 88349
доля событий train-куки вне окна: 0.171


## 3. Признаки

Идея — описать не только объём активности, но и её характер: ритм, разнообразие,
технические следы.

- **Объём и разнообразие**: число событий и их типов, уникальные объявления /
  категории / локации / запросы, глубина выдачи, счётчики по типам событий и доли.
- **Повторы**: `item_repeat_rate` — повторные просмотры карточки,
  `item_revisit_frac` — доля событий по уже виденным объявлениям.
- **Ритм**: статистики интервалов между событиями, длительность, темп.
- **Технические следы**: число платформ и user-agent у куки, тип клиента по
  user-agent (headless-браузер, http-библиотека, приложение Авито, обычные браузеры).
- **Курсор**: разброс `pointer_x` и доля точек на краю экрана.
- **Возраст куки** на начало окна.

Пробовал и убрал: флаги `has_*` (дублируют счётчики событий), энтропию и число
часов активности, долю pro-продавцов, долю событий с курсором, разброс `pointer_y`,
доли интервалов по диапазонам и сессии — на валидации пользы от них не было. Флаг
«ключевые слова автоматизации в user-agent» заменил типом клиента: `okhttp` есть и
в user-agent официального Android-приложения Авито, и флаг ловил обычных людей.

In [8]:
EVENT_IDS = [100, 200, 210, 220, 300, 301, 303, 400, 500]
UA_FAMILIES = [
    "headless_chrome",
    "http_library",
    "avito_app",
    "yabrowser",
    "firefox",
    "mobile_safari",
    "android_chrome",
    "desktop_chrome",
]
SCREEN_W, SCREEN_H = 1920, 1080  # максимальные координаты курсора в данных


def ua_family(ua):
    # тип клиента вместо сырой строки user-agent (версии браузеров — шум)
    if "HeadlessChrome" in ua:
        return "headless_chrome"
    if ua.startswith("Avito/"):
        return "avito_app"  # приложение Авито; внутри есть okhttp, но это не бот
    if not ua.startswith("Mozilla/"):
        return "http_library"  # curl, python-requests, Go-http-client, ...
    if "YaBrowser" in ua:
        return "yabrowser"
    if "Firefox" in ua:
        return "firefox"
    if "iPhone" in ua:
        return "mobile_safari"
    if "Android" in ua:
        return "android_chrome"
    return "desktop_chrome"


def build_features(events_win, meta):
    ev = events_win.sort_values("event_ts").copy()
    ev["platform_norm"] = ev["platform"].str.lower().replace({"iphone": "ios"})
    ev["ua_family"] = ev["user_agent"].map(ua_family)
    ev["pointer_edge"] = ev["pointer_x"].isin([0, SCREEN_W]) | ev["pointer_y"].isin(
        [0, SCREEN_H]
    )

    rows = []
    for cid, g in ev.groupby("cookie_id"):
        n = len(g)
        ts = g["event_ts"].values
        if n >= 2:
            iat = np.diff(ts).astype("timedelta64[s]").astype(float)
            iat_mean, iat_std = iat.mean(), iat.std()
            iat_cv = iat_std / (iat_mean + 1e-6)
            iat_min, iat_median = iat.min(), float(np.median(iat))
            span_sec = (ts[-1] - ts[0]) / np.timedelta64(1, "s")
        else:
            # одно событие — интервалов нет
            iat_mean = iat_std = iat_cv = iat_min = iat_median = np.nan
            span_sec = 0.0

        # доля событий по уже виденным объявлениям
        item_ids = g["item_id"].dropna()
        item_revisit_frac = (
            1 - item_ids.nunique() / len(item_ids) if len(item_ids) else np.nan
        )

        row = {
            "cookie_id": cid,
            "n_events": n,
            "n_eid_distinct": g["eid"].nunique(),
            "item_nunique": g["item_id"].nunique(),
            "item_view_nunique": g.loc[g["eid"] == 200, "item_id"].nunique(),
            "category_nunique": g["item_category"].nunique(),
            "location_nunique": g["item_location"].nunique(),
            "search_query_nunique": g["search_query"].nunique(),
            "search_page_max": g["search_page"].max(),
            "search_page_mean": g["search_page"].mean(),
            "platform_nunique": g["platform_norm"].nunique(),
            "ua_nunique": g["user_agent"].nunique(),
            "ua_family": g["ua_family"].mode().iloc[0],
            "iat_mean": iat_mean,
            "iat_std": iat_std,
            "iat_cv": iat_cv,
            "iat_min": iat_min,
            "iat_median": iat_median,
            "span_sec": span_sec,
            "events_per_min_active": n / (span_sec / 60 + 1),
            "pointer_x_std": g["pointer_x"].std(),  # NaN, если курсора нет совсем
            "pointer_edge_share": g.loc[g["pointer_x"].notna(), "pointer_edge"].mean(),
            "item_revisit_frac": item_revisit_frac,
        }
        row.update({f"cnt_eid_{e}": c for e, c in g["eid"].value_counts().items()})
        rows.append(row)

    feat = meta[["cookie_id", "cookie_created_at", "window_start_ts"]].merge(
        pd.DataFrame(rows), on="cookie_id", how="left"
    )
    cnt_cols = [f"cnt_eid_{e}" for e in EVENT_IDS]
    feat[cnt_cols] = feat.reindex(columns=cnt_cols).fillna(
        0
    )  # тип события не встречался -> 0
    # одинаковый список категорий в train и test
    feat["ua_family"] = pd.Categorical(feat["ua_family"], categories=UA_FAMILIES)

    feat["item_view_ratio"] = feat["cnt_eid_200"] / feat["n_events"]
    feat["search_ratio"] = feat["cnt_eid_100"] / feat["n_events"]
    feat["photo_swipe_per_item"] = feat["cnt_eid_210"] / feat["item_nunique"].replace(
        0, np.nan
    )
    # повторные просмотры карточки (уникальные id — только среди item_view)
    feat["item_repeat_rate"] = 1 - feat["item_view_nunique"] / feat[
        "cnt_eid_200"
    ].replace(0, np.nan)
    feat["account_age_days"] = (
        feat["window_start_ts"] - feat["cookie_created_at"]
    ).dt.total_seconds() / 86400
    return feat.drop(columns=["cookie_created_at", "window_start_ts"])


Xtr = build_features(ev_tr, train)
Xte = build_features(ev_te, test)
ytr = train.set_index("cookie_id").loc[Xtr.cookie_id, "target"].values

print("признаков куки:", Xtr.shape[1] - 1)
print(Xtr.shape, Xte.shape)

признаков куки: 36
(11091, 37) (4909, 37)


### Общие объявления

Боты разных кук обходят общий пул объявлений, а люди смотрят каждый своё. Отсюда
две группы признаков:

- **популярность** — сколько других кук с окном в тот же день или раньше видели
  те же объявления. Метки не нужны, тестовые куки тоже считаются соседями. Делим на
  число таких кук, чтобы признак не рос со временем;
- **репутация** — сколько известных ботов и людей видели те же объявления. Берутся
  только метки кук с окном строго раньше `cutoff`: для train — дня окна самой куки,
  на валидации — даты отсечки, для теста — все метки train (они все из более ранних
  дней). Так признак доступен к концу окна, как в проде, где разметка прошлых дней
  уже есть.

In [9]:
meta = pd.concat([train.drop(columns="target"), test], ignore_index=True)
window_day = meta.set_index("cookie_id")["window_start_ts"]
labels = train.set_index("cookie_id")["target"]

# уникальные пары кука–объявление по событиям внутри окон (train и test)
pairs = pd.concat([ev_tr, ev_te]).dropna(subset=["item_id"])[["cookie_id", "item_id"]]
pairs = pairs.drop_duplicates()
pairs["day"] = pairs["cookie_id"].map(window_day)


def item_popularity_features(pairs):
    cnt = pairs.groupby(["item_id", "day"]).size().rename("n").reset_index()
    cnt = cnt.sort_values(["item_id", "day"])
    cnt["cum"] = cnt.groupby("item_id")["n"].cumsum()  # куки с окном в дни <= day
    p = pairs.merge(cnt[["item_id", "day", "cum"]], on=["item_id", "day"])
    p["others"] = p["cum"] - 1  # минус сама кука
    cookies_upto_day = window_day.value_counts().sort_index().cumsum()
    p["others_rate"] = p["others"] / p["day"].map(cookies_upto_day) * 1000
    g = p.groupby("cookie_id")
    return pd.DataFrame(
        {
            "item_pop_mean": g["others_rate"].mean(),
            "item_pop_max": g["others_rate"].max(),
            "item_shared_share": g["others"].agg(lambda s: (s > 0).mean()),
        }
    )


def item_label_features(cutoff):
    # cutoff: Series cookie_id -> дата; берём метки только кук с окном < cutoff
    lab = pairs[pairs["cookie_id"].isin(labels.index)].copy()
    lab["y"] = lab["cookie_id"].map(labels)
    lab = lab.groupby(["item_id", "day"])["y"].agg(["sum", "size"]).reset_index()
    lab = lab.sort_values(["item_id", "day"])
    lab["cum_bot"] = lab.groupby("item_id")["sum"].cumsum()
    lab["cum_all"] = lab.groupby("item_id")["size"].cumsum()
    lab["available_from"] = lab["day"] + pd.Timedelta(days=1)  # метка дня d известна с d+1

    q = pairs[["cookie_id", "item_id"]].copy()
    q["cutoff"] = q["cookie_id"].map(cutoff)
    # для каждой пары — последний накопленный счётчик с available_from <= cutoff
    m = pd.merge_asof(
        q.sort_values("cutoff"),
        lab[["item_id", "available_from", "cum_bot", "cum_all"]].sort_values("available_from"),
        left_on="cutoff",
        right_on="available_from",
        by="item_id",
        direction="backward",
    )
    m[["cum_bot", "cum_all"]] = m[["cum_bot", "cum_all"]].fillna(0)
    g = m.groupby("cookie_id")
    f = pd.DataFrame(
        {
            "nb_bot_views": g["cum_bot"].sum(),
            "nb_human_views": g["cum_all"].sum() - g["cum_bot"].sum(),
            "nb_items_with_bot": g["cum_bot"].agg(lambda s: (s > 0).mean()),
            "nb_bot_max": g["cum_bot"].max(),
        }
    )
    # доля ботов среди размеченных «соседей»; NaN, если таких нет
    total = f["nb_bot_views"] + f["nb_human_views"]
    f["nb_bot_ratio"] = f["nb_bot_views"] / total.replace(0, np.nan)
    return f


def with_labels(X, cutoff):
    return X.join(item_label_features(cutoff), on="cookie_id")


F_pop = item_popularity_features(pairs)
Xtr = Xtr.join(F_pop, on="cookie_id")
Xte = Xte.join(F_pop, on="cookie_id")
LAB_COLS = ["nb_bot_views", "nb_human_views", "nb_items_with_bot", "nb_bot_max", "nb_bot_ratio"]
FEATURES = [c for c in Xtr.columns if c != "cookie_id"] + LAB_COLS
NUM_FEATURES = [c for c in FEATURES if c != "ua_family"]  # для RF и логрега
print("всего признаков:", len(FEATURES))

# мотивация: у ботов объявления заметно чаще «общие» с другими куками
print(Xtr.assign(target=ytr).groupby("target")[["item_shared_share", "item_pop_mean"]].mean().round(3))

всего признаков: 44
        item_shared_share  item_pop_mean
target                                  
0                   0.552          0.169
1                   0.689          0.298


Пропуски (нет курсора, одно событие в окне, нет просмотров объявлений) оставляю как
NaN, а не 0: «нет данных» — это не «ноль». `HistGradientBoostingClassifier` умеет
работать с NaN и с категориальным `ua_family` сам.

## 4. Валидация

Обучаемся на ранних окнах, проверяем на поздних; три точки отсечения, метрика из
`metric.py`.

In [10]:
from metric import precision_at_recall
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

window_start = window_day.loc[Xtr.cookie_id]  # день окна для каждой строки train
CUTS = ["2026-04-13", "2026-04-15", "2026-04-17"]
BASELINE_COLS = ["n_events", "item_nunique"]

for cut in CUTS:
    is_valid = window_start.ge(cut).values
    print(
        f"cut {cut}: train={(~is_valid).sum()}, valid={is_valid.sum()} (ботов в valid: {ytr[is_valid].sum()})"
    )

# признаки-метки для каждой отсечки: валидационные куки видят только метки до cut
X_CUT = {cut: with_labels(Xtr, window_day.clip(upper=pd.Timestamp(cut))) for cut in CUTS}


def time_cv(make_model, cols):
    # обучение на окнах до cut, проверка на окнах >= cut
    scores = []
    for cut in CUTS:
        X = X_CUT[cut]
        is_valid = window_start.ge(cut).values
        model = make_model().fit(X.loc[~is_valid, cols], ytr[~is_valid])
        scores.append(
            precision_at_recall(
                ytr[is_valid], model.predict_proba(X.loc[is_valid, cols])[:, 1]
            )
        )
    return np.array(scores)


def make_hgb(**params):
    # без ранней остановки: иначе на всём train (>10k строк) sklearn включит её сам
    return HistGradientBoostingClassifier(
        early_stopping=False, random_state=RANDOM_STATE, **params
    )


def with_imputer(*steps):
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), *steps)


CANDIDATES = {
    "baseline (RF, 2 признака quickstart)": (
        lambda: RandomForestClassifier(
            n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE
        ),
        BASELINE_COLS,
    ),
    "LogisticRegression, все признаки": (
        lambda: with_imputer(StandardScaler(), LogisticRegression(max_iter=2000)),
        NUM_FEATURES,
    ),
    "RandomForest, все признаки": (
        lambda: with_imputer(
            RandomForestClassifier(
                n_estimators=500,
                min_samples_leaf=3,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            )
        ),
        NUM_FEATURES,
    ),
    "HistGradientBoosting, без общих объявлений": (
        lambda: make_hgb(
            max_depth=3,
            learning_rate=0.05,
            max_iter=300,
            min_samples_leaf=20,
            l2_regularization=1.0,
        ),
        [c for c in FEATURES if c not in list(F_pop.columns) + LAB_COLS],
    ),
    "HistGradientBoosting, все признаки": (
        lambda: make_hgb(
            max_depth=3,
            learning_rate=0.05,
            max_iter=300,
            min_samples_leaf=20,
            l2_regularization=1.0,
        ),
        FEATURES,
    ),
}

print()
print(f"{'модель':44s}" + "".join(f"{c:>12s}" for c in CUTS) + f"{'среднее':>10s}")
for name, (make_model, cols) in CANDIDATES.items():
    s = time_cv(make_model, cols)
    print(f"{name:44s}" + "".join(f"{x:12.4f}" for x in s) + f"{s.mean():10.4f}")

cut 2026-04-13: train=5930, valid=5161 (ботов в valid: 408)
cut 2026-04-15: train=7599, valid=3492 (ботов в valid: 285)
cut 2026-04-17: train=9140, valid=1951 (ботов в valid: 160)

модель                                        2026-04-13  2026-04-15  2026-04-17   среднее
baseline (RF, 2 признака quickstart)              0.0931      0.0912      0.0957    0.0933
LogisticRegression, все признаки                  0.7204      0.7353      0.7063    0.7206
RandomForest, все признаки                        0.8800      0.8632      0.8129    0.8521
HistGradientBoosting, без общих объявлений        0.7586      0.7663      0.8235    0.7828
HistGradientBoosting, все признаки                0.8683      0.8772      0.8819    0.8758


Бустинг сильно лучше baseline из quickstart (~0.87 против ~0.09) и лучше
RandomForest (~0.85) и логрега (~0.72). Главный прирост дают признаки по общим
объявлениям: без них тот же бустинг даёт ~0.78. Гиперпараметры выбираю по среднему
по трём отсечкам.

In [11]:
CONFIGS = [
    dict(
        max_depth=3,
        learning_rate=0.05,
        max_iter=300,
        min_samples_leaf=20,
        l2_regularization=1.0,
    ),
    dict(
        max_depth=3,
        learning_rate=0.08,
        max_iter=250,
        min_samples_leaf=30,
        l2_regularization=1.0,
    ),
    dict(
        max_depth=4,
        learning_rate=0.05,
        max_iter=400,
        min_samples_leaf=15,
        l2_regularization=1.0,
    ),
    dict(
        max_depth=4,
        learning_rate=0.03,
        max_iter=500,
        min_samples_leaf=20,
        l2_regularization=2.0,
    ),
    dict(
        max_depth=5,
        learning_rate=0.03,
        max_iter=400,
        min_samples_leaf=10,
        l2_regularization=1.0,
    ),
]

results = []
for cfg in CONFIGS:
    s = time_cv(lambda: make_hgb(**cfg), FEATURES)
    results.append((cfg, s.mean()))
    print(
        f"depth={cfg['max_depth']} lr={cfg['learning_rate']:<5} iter={cfg['max_iter']:<4} "
        f"leaf={cfg['min_samples_leaf']:<3} l2={cfg['l2_regularization']:<4} -> "
        + " ".join(f"{x:.4f}" for x in s)
        + f"  среднее {s.mean():.4f}"
    )

BEST_CFG = max(results, key=lambda t: t[1])[0]
print("\nвыбрана конфигурация:", BEST_CFG)

depth=3 lr=0.05  iter=300  leaf=20  l2=1.0  -> 0.8683 0.8772 0.8819  среднее 0.8758
depth=3 lr=0.08  iter=250  leaf=30  l2=1.0  -> 0.8496 0.8788 0.8828  среднее 0.8704
depth=4 lr=0.05  iter=400  leaf=15  l2=1.0  -> 0.8754 0.8855 0.8819  среднее 0.8809
depth=4 lr=0.03  iter=500  leaf=20  l2=2.0  -> 0.8777 0.8811 0.8819  среднее 0.8802
depth=5 lr=0.03  iter=400  leaf=10  l2=1.0  -> 0.8727 0.8739 0.8819  среднее 0.8762

выбрана конфигурация: {'max_depth': 4, 'learning_rate': 0.05, 'max_iter': 400, 'min_samples_leaf': 15, 'l2_regularization': 1.0}


## 5. Важность признаков

Permutation importance на валидации с отсечкой 15 апреля.

In [12]:
from sklearn.inspection import permutation_importance

X = X_CUT["2026-04-15"]
is_valid = window_start.ge("2026-04-15").values
model = make_hgb(**BEST_CFG)
model.fit(X.loc[~is_valid, FEATURES], ytr[~is_valid])

perm = permutation_importance(
    model,
    X.loc[is_valid, FEATURES],
    ytr[is_valid],
    n_repeats=8,
    random_state=RANDOM_STATE,
    scoring="roc_auc",
    n_jobs=-1,
)
importance = pd.Series(perm.importances_mean, index=FEATURES).sort_values(
    ascending=False
)
print(importance.head(15).round(4).to_string())

pointer_x_std         0.0240
iat_median            0.0219
pointer_edge_share    0.0156
category_nunique      0.0095
iat_min               0.0077
location_nunique      0.0053
item_revisit_frac     0.0042
nb_items_with_bot     0.0026
n_events              0.0017
iat_mean              0.0016
search_page_mean      0.0014
account_age_days      0.0012
ua_family             0.0010
nb_bot_ratio          0.0009
cnt_eid_400           0.0009


Больше всего модель опирается на курсор (разброс и точки на краю экрана), интервалы
между событиями, разнообразие категорий/локаций и повторные обращения к объявлениям.
У признаков по общим объявлениям важность по отдельности небольшая: они дублируют
друг друга, и перемешивание одного почти не мешает модели. Вместе они дают около
+0.1 к метрике (строка «без общих объявлений» в таблице выше).

## 6. Сабмит

Финальная модель — на всём train.

In [13]:
# признаки-метки: для train — метки дней до окна куки, для теста — все метки train
X_train = with_labels(Xtr, window_day)
X_test = with_labels(Xte, window_day)

final_model = make_hgb(**BEST_CFG)
final_model.fit(X_train[FEATURES], ytr)

test_scores = final_model.predict_proba(X_test[FEATURES])[:, 1]

submission = pd.DataFrame(
    {
        "cookie_id": Xte["cookie_id"],
        "score": test_scores,
    }
)

assert len(submission) == len(test)
assert submission["cookie_id"].is_unique
assert set(submission["cookie_id"]) == set(test["cookie_id"])
assert submission["score"].between(0, 1).all()

submission.to_csv("submission.csv", index=False)
print(submission.shape)
print(submission.head())
print("\nраспределение score:")
print(submission["score"].describe())

(4909, 2)
             cookie_id     score
0  ck_315fb710a0e371e7  0.001775
1  ck_a76ee3b3e3e522fd  0.161802
2  ck_94c9a4d382689e82  0.030835
3  ck_8eaf9509ad9462a0  0.000769
4  ck_9a88a5a989cb5bc6  0.077133

распределение score:
count    4909.000000
mean        0.088051
std         0.233110
min         0.000022
25%         0.002726
50%         0.008239
75%         0.028486
max         0.999385
Name: score, dtype: float64


## 7. Описание подхода

**Данные.** Удалил дубликаты событий, считаю признаки только по событиям внутри окна.

**Признаки.** 44 признака:
- поведение куки: объём и разнообразие действий, повторы объявлений, ритм (интервалы
  между событиями), курсор (разброс, точки на краю экрана), тип клиента по
  user-agent, возраст куки;
- общие объявления: боты обходят общий пул объявлений, поэтому считаю, сколько
  других кук видели те же объявления (популярность) и сколько среди них известных
  ботов и людей (репутация). Для репутации беру только метки кук с окном строго
  раньше: на валидации — до даты отсечки, для теста — метки train.

**Валидация.** По времени, три точки отсечения (13, 15, 17 апреля).

**Модель.** `HistGradientBoostingClassifier` (sklearn). Средний P@R0.7 на валидации
≈ 0.88; без признаков по общим объявлениям ≈ 0.78; baseline из quickstart ≈ 0.09,
RandomForest ≈ 0.85, логрег ≈ 0.72.

**Ограничения.** Ботов в валидации немного (160–408), поэтому оценка шумная.
Репутация объявлений работает, пока пул объявлений, интересных парсерам, не
меняется. На тесте последние известные метки отстают на 1–7 дней — так же, как на
первой отсечке валидации, так что оценка это учитывает.

**Воспроизводимость.** Только numpy, pandas, scikit-learn. `random_state=0`,
Python 3.14, версии в `requirements.txt`. Запускать из корня архива (рядом с
`metric.py` и `data/`).